# Exercices XP
Dernière mise à jour : 30 juillet 2025

👩‍🏫 👩🏿‍🏫 Ce que vous apprendrez
Comment créer des interfaces Gradio simples et multifonctionnelles en utilisant Interfaceet Blocks.
Comment gérer la mémoire conversationnelle et l'interactivité dans les applications Streamlit.
Comment créer et afficher du contenu Streamlit dynamique à l'aide de graphiques, de widgets et de conditions.
Comment concevoir des itinéraires FastAPI avec une logique de routage intelligente basée sur des mots-clés.


🛠️ Ce que vous allez créer
Une application multi-outils et une calculatrice alimentées par Gradio.
Un chatbot Streamlit avec mémoire de session.
Une interface utilisateur dynamique et multiformat dans Streamlit.
Un backend FastAPI intelligent qui renvoie des réponses contextuelles.


# 1. Enjeux pédagogiques globaux

Cet exercice propose une exploration large et progressive des frameworks d’interface Python les plus utilisés aujourd’hui pour prototyper et déployer des applications interactives d’IA et de data science : Gradio, Streamlit et FastAPI. L’objectif est d’apprendre à créer rapidement :

* des outils multi-fonctions,

* des interfaces réactives,

* des chatbots à mémoire,

* des tableaux de bord dynamiques,

* des APIs back-end intelligentes.

À chaque étape, l’accent est mis sur l’interactivité, la gestion d’état (mémoire de session), la modularité des composants et la capacité à scripter des logiques conditionnelles selon les besoins métier.

# 2. Exercice 1 : Gradio - Interface multifonctions

But :
Créer une interface utilisateur simple et polyvalente qui permet de sélectionner, via un menu déroulant, plusieurs fonctionnalités différentes accessibles via une même interface.

Concepts abordés :

Gradio Interface : abstraction simple pour créer des applications web autour de fonctions Python.

Menus déroulants (dropdown) et entrées conditionnelles (texte, nombre) selon l’option choisie.

Mappage fonctionnel : choix utilisateur → fonction Python appelée.

Démo interactive : examples pour montrer chaque fonctionnalité possible.

Compétences visées :

Concevoir une interface réactive et dynamique sans écrire de frontend HTML/CSS.

Comprendre comment exposer plusieurs outils à travers une même interface utilisateur.

Spécificités techniques :

Gestion du typage dynamique (textbox pour du texte, number pour du calcul).

Intégration d’exemples (cas d’usage pédagogiques).

Structuration claire du code backend pour que chaque choix appelle le bon comportement.

In [ ]:
# Exercice 1 : Interface Gradio – Boîte à outils multifonctions

# 1. Import des modules nécessaires
import gradio as gr

# 2. Définition des trois fonctions principales :
def greet(name):
    """Retourne une salutation personnalisée."""
    return f"Hello, {name}!"

def echo(text):
    """Répète ce que l'utilisateur a saisi."""
    return f"You said: {text}"

def square_number(number):
    """Calcule le carré d'un nombre donné."""
    return number ** 2

# 3. Logique centrale pour le choix dynamique de la fonction et de l'input :
def multi_tool(choice, user_input):
    """
    Appelle la bonne fonction selon le choix utilisateur.
    Gère dynamiquement le typage de l'input.
    """
    if choice == "Greet":
        return greet(user_input)
    elif choice == "Echo":
        return echo(user_input)
    elif choice == "Square a Number":
        # Conversion forcée si l'input est du texte
        try:
            number = float(user_input)
        except Exception:
            return "Erreur : veuillez entrer un nombre."
        return square_number(number)
    else:
        return "Choix inconnu."

# 4. Construction de l'interface Gradio
# On crée une fonction de wrapper pour changer dynamiquement le composant d'entrée
def dynamic_interface(choice):
    if choice == "Square a Number":
        return gr.Number(label="Entrez un nombre à mettre au carré")
    else:
        return gr.Textbox(label="Votre saisie")

with gr.Blocks() as demo:
    gr.Markdown("## Multi-tool Gradio Interface\nChoisissez une fonction et entrez une valeur adaptée :")
    choice = gr.Dropdown(choices=["Greet", "Echo", "Square a Number"], label="Choisissez une fonction", value="Greet")
    input_box = gr.Textbox(label="Votre saisie")  # Défaut: textbox

    output = gr.Textbox(label="Résultat")

    def update_input_box(choice):
        # Permet de changer dynamiquement le type d'input affiché
        if choice == "Square a Number":
            return gr.Number(label="Entrez un nombre à mettre au carré")
        else:
            return gr.Textbox(label="Votre saisie")

    # Rendu dynamique de la boîte d'entrée selon le choix
    choice.change(update_input_box, inputs=choice, outputs=input_box)

    # Bouton de validation
    btn = gr.Button("Valider")
    btn.click(
        fn=multi_tool,
        inputs=[choice, input_box],
        outputs=output
    )

    # Exemples pour chaque fonctionnalité
    gr.Examples(
        examples=[
            ["Greet", "Alice"],
            ["Echo", "Bonjour Gradio"],
            ["Square a Number", 7]
        ],
        inputs=[choice, input_box],
        outputs=output
    )

# 5. Lancement de l'application
demo.launch()

# === EXPLICATIONS ===
# - L'utilisateur choisit la fonction désirée via la liste déroulante.
# - Le champ de saisie s'adapte au choix (texte ou nombre).
# - Lorsqu'on valide, la fonction adéquate est appelée et le résultat s'affiche.
# - Les exemples préremplissent les champs pour guider l'utilisateur.


# 3. Exercice 2 : Gradio Blocks – Calculatrice personnalisée

But :
Passer du modèle “interface unique” à une composition flexible d’éléments graphiques à l’aide de Gradio Blocks, pour créer des outils plus complexes et personnalisés (ici, une calculatrice avec choix d’opération).

Concepts abordés :

gr.Blocks : approche composantielle et réactive (layout par ligne, colonne, row, etc.).

Câblage d’événements (méthode .click() pour relier bouton et fonction).

Inputs multiples (deux zones pour nombres), radio (choix de l’opération), output (label), affichage Markdown pour instructions.

Compétences visées :

Maîtriser l’API Blocks de Gradio pour une mise en page avancée.

Concevoir des workflows utilisateurs à étapes (input → action → output).

Utiliser des événements pour synchroniser l’UI et le backend.

Spécificités techniques :

Customisation du layout (alignement horizontal, logique conditionnelle).

Interaction explicite par bouton pour valider l’action (meilleur contrôle que le mode auto).

In [ ]:
# Exercice 2 : Gradio Blocks – Calculatrice simple en deux étapes

import gradio as gr

# 1. Fonction principale de calcul
def calculer(num1, num2, operation):
    """Effectue une addition ou une multiplication selon le choix."""
    if operation == "Add":
        result = num1 + num2
    elif operation == "Multiply":
        result = num1 * num2
    else:
        return "Operation inconnue."
    return f"Résultat : {result}"

# 2. Construction de l'interface avec gr.Blocks
with gr.Blocks() as calculatrice:
    # Affichage Markdown d'instructions
    gr.Markdown("## Calculatrice simple\nSaisissez deux nombres, choisissez une opération puis cliquez sur le bouton.")
    
    # Mise en page personnalisée : deux inputs côte à côte
    with gr.Row():
        entree1 = gr.Number(label="Nombre 1")
        entree2 = gr.Number(label="Nombre 2")
    
    # Choix de l'opération (radio)
    operation = gr.Radio(choices=["Add", "Multiply"], label="Opération", value="Add")
    
    # Bouton de validation
    bouton = gr.Button("Calculer")
    
    # Zone de sortie (affichage du résultat)
    sortie = gr.Label(label="Résultat")

    # Lien bouton -> fonction -> sortie
    bouton.click(
        fn=calculer,
        inputs=[entree1, entree2, operation],
        outputs=sortie
    )

# 3. Lancement de l'application Gradio
calculatrice.launch()

# === EXPLICATIONS ===
# - gr.Blocks permet une disposition flexible (layout) des composants.
# - gr.Row() aligne les deux zones de saisie horizontalement.
# - gr.Radio propose le choix de l'opération (addition ou multiplication).
# - L'utilisateur clique sur "Calculer" pour afficher le résultat dans gr.Label.
# - Les instructions sont affichées en haut en Markdown pour clarifier l'usage.


# 4. Exercice 3 : Streamlit Chat – Bot à mémoire d’état

But :
Créer une interface de chatbot dans Streamlit capable de mémoriser l’historique des échanges sur la session de l’utilisateur, illustrant la gestion de la mémoire conversationnelle côté UI.

Concepts abordés :

st.chat_message() : affichage des messages dans une interface de chat.

st.session_state : gestion de l’état et de l’historique sur la durée de la session utilisateur.

UI avec ou sans état : différence fondamentale pour la persistance des données utilisateur.

Compétences visées :

Initialiser et manipuler des variables d’état dans une application web (gestion mémoire conversationnelle).

Afficher dynamiquement l’historique des échanges.

Implémenter un bouton pour réinitialiser l’historique (reset complet de session_state).

Spécificités techniques :

Boucler sur l’historique pour ré-afficher chaque message à chaque render de la page.

Synchroniser les entrées utilisateur, les réponses bot, et la logique d’effacement.

In [ ]:
!pip install streamlit


In [ ]:
# Exercice 3 : Streamlit Chat - Bot de mémoire à état

# Pour exécuter ce code, sauvegarde-le dans un fichier nommé par exemple `chatbot_streamlit.py`,
# puis exécute en console : streamlit run chatbot_streamlit.py

import streamlit as st

# 1. Titre de l’application
st.title("Chatbot à mémoire de session")

# 2. Initialisation de la mémoire de session si inexistante
if "chat_history" not in st.session_state:
    st.session_state["chat_history"] = []

# 3. Affichage de l’historique des messages
for sender, message in st.session_state["chat_history"]:
    st.chat_message(sender).write(message)

# 4. Saisie utilisateur
user_input = st.chat_input("Votre message ici...")

# 5. Si un nouveau message est saisi, stockage et réponse du bot (écho)
if user_input:
    # Ajout du message utilisateur à l’historique
    st.session_state["chat_history"].append(("user", user_input))
    # Génération de la réponse (ici simple écho)
    bot_reply = f"Bot: {user_input}"
    st.session_state["chat_history"].append(("bot", bot_reply))
    # Affichage immédiat de la réponse du bot
    st.chat_message("bot").write(bot_reply)

# 6. Bouton pour effacer la conversation
if st.button("Effacer la conversation"):
    st.session_state["chat_history"] = []
    st.experimental_rerun()

# === EXPLICATIONS ===
# - Toute la mémoire de la conversation est stockée dans st.session_state["chat_history"].
# - L’historique est réaffiché à chaque render pour garantir la persistance de la mémoire.
# - st.chat_input offre une saisie propre et st.chat_message permet un affichage stylé façon chat.
# - Le bouton "Effacer la conversation" réinitialise l’historique et relance la page.


# 5. Exercice 4 : Streamlit - Interface multi-formats et logique conditionnelle

But :
Développer un mini-tableau de bord interactif qui affiche différents types de contenu (graphiques, code, JSON, images) selon les actions de l’utilisateur.

Concepts abordés :

st.title, st.radio : construction de layouts simples mais dynamiques.

st.line_chart : génération et visualisation de données aléatoires.

st.code, st.json : rendu conditionnel de contenu formaté (code Python vs JSON).

st.image : chargement d’images externes.

Utilisation de NumPy pour la génération de données dynamiques.

Compétences visées :

Structurer un dashboard simple mais adaptatif.

Mettre en œuvre des logiques conditionnelles (if/else) pour le rendu dynamique.

Gérer différents types de contenus dans une interface réactive.

Spécificités techniques :

Manipulation de données et génération à la volée (NumPy).

Rendu visuel adapté selon le choix utilisateur.

In [ ]:
!pip install streamlit numpy

In [ ]:
# Exercice 4 : Streamlit - Interface utilisateur simplifiée + logique conditionnelle

import streamlit as st
import numpy as np

# 1. Titre de l'application
st.title("Mini-tableau de bord Streamlit dynamique")

# 2. Générer et afficher un graphique de données aléatoires
data = np.random.randn(30, 3)  # 30 lignes, 3 séries aléatoires
st.line_chart(data, use_container_width=True)

# 3. Choix utilisateur pour l'affichage du contenu
affichage = st.radio("Afficher :", options=["Show Code", "Show JSON"], horizontal=True)

# 4. Rendu conditionnel : bloc code ou bloc JSON
if affichage == "Show Code":
    code_example = """
def hello(name):
    print(f"Hello, {name}!")
"""
    st.code(code_example, language="python")
elif affichage == "Show JSON":
    json_example = {
        "status": "ok",
        "data": [1, 2, 3],
        "message": "Exemple JSON"
    }
    st.json(json_example)

# 5. Bonus : affichage d'une image publique
image_url = "https://upload.wikimedia.org/wikipedia/commons/3/3f/Fronalpstock_big.jpg"
st.image(image_url, caption="Image publique (Wikimedia)", use_column_width=True)

# === EXPLICATIONS ===
# - Le titre et le graphique s'affichent systématiquement au chargement.
# - L'utilisateur choisit d'afficher soit un bloc de code, soit un JSON formaté.
# - L'image publique est toujours affichée en bonus.
# - Tout est réactif : changer l'option actualise le contenu affiché.


# 6. Exercice 5 : FastAPI – Backend intelligent avec logique contextuelle

But :
Implémenter un service backend avec FastAPI capable de répondre différemment selon l’analyse sémantique d’un message reçu.

Concepts abordés :

FastAPI : création d’API REST modernes et rapides en Python.

Décorateurs de route (@app.post), Pydantic pour le typage et la validation des entrées JSON.

Logique de routage intelligente basée sur des mots-clés détectés dans la requête.

Test local d’API avec curl/Postman ou des clients Python.

Compétences visées :

Définir et documenter des endpoints robustes.

Gérer le routage conditionnel côté backend (ex : redirection vers un calculateur, une horloge, etc.).

Valider et traiter dynamiquement des requêtes JSON entrantes.

Spécificités techniques :

Construction de schémas de requête/réponse avec Pydantic.

Tests manuels d’API (debug, validation des chemins, réponses adaptées).

In [ ]:
!pip install fastapi uvicorn

In [ ]:
# Exercice 5 : FastAPI – Répondeur intelligent

# 1. Imports nécessaires
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn

# 2. Définition du modèle de requête (entrée JSON attendue)
class MessageRequest(BaseModel):
    message: str

# 3. Création de l'application FastAPI
app = FastAPI()

# 4. Définition du endpoint POST avec logique contextuelle
@app.post("/respond")
def respond(request: MessageRequest):
    msg = request.message.lower()
    # Routage intelligent selon mots-clés détectés
    if "math" in msg:
        return {"response": "Using calculator tool..."}
    elif "date" in msg:
        return {"response": "Fetching current date..."}
    else:
        return {"response": "Default LLM response."}

# 5. Lancement du serveur si exécution directe
if __name__ == "__main__":
    # Uvicorn lance l'app sur http://127.0.0.1:8000
    uvicorn.run("fastapi_repondeur:app", host="127.0.0.1", port=8000, reload=True)

# === EXPLICATIONS ===
# - L'utilisateur envoie une requête POST sur /respond avec un JSON {"message": "..."}.
# - Selon la présence des mots-clés "math" ou "date" dans le message, la réponse change.
# - Sinon, la réponse générique "Default LLM response." est renvoyée.
# - Test possible via curl, Postman, httpie ou requests.


http POST http://127.0.0.1:8000/respond message="What's 5+5? (math)"